# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****f954
LANGSMITH_API_KEY=****c830
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="openai:cohere/north-mini-code:free", 
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

Hey there! I’m doing great—basically the human equivalent of a full‑stack dev, except I swap out the bug reports for punchlines and the code for clever comedi‑logic. 😄

What brings you here today? Ready to hear a joke that’s “full‑stack” enough to run on both your brain and your heart? Or maybe we should debug some of your own?


In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey there! I’m doing great—basically the human equivalent of a full‑stack dev, except I swap out the bug reports for punchlines and the code for clever comedi‑logic. 😄

What brings you here today? Ready to hear a joke that’s “full‑stack” enough to run on both your brain and your heart? Or maybe we should debug some of your own?



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
agent = create_agent(
    model="openai:cohere/north-mini-code:free",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

A diamond pulse where legends rise,  
a crack of willow, leather’s sigh.  
Nine innings, hearts alight,  
the eighth‑inning thrill, the final fight.  
A glove of silk, a fastball swift—  
baseball, timeless, boundless drift.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Feet strike the track fast  
Lightning flashes, breath in fire  
Gold snaps the finish


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
agent = create_agent(
    model="openai:cohere/north-mini-code:free",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [12]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 First strike takes aim
Whistle blows in midnight air
Victory or fall


In [13]:
result["messages"][-1].content

"Here's a sports haiku for you:\n\nFirst strike takes aim\nWhistle blows in midnight air  \nVictory or fall\n\nThis Haiku has been checked and confirmed to have exactly 3 lines. \n\nWould you like me to write another Haiku for you?"

In [14]:
print(len(result["messages"]))

4


In [15]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (check_haiku_lines_5bnzecnk8agg)
 Call ID: check_haiku_lines_5bnzecnk8agg
  Args:
    text: First strike takes aim
Whistle blows in midnight air
Victory or fall
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Here's a sports haiku for you:

First strike takes aim
Whistle blows in midnight air  
Victory or fall

This Haiku has been checked and confirmed to have exactly 3 lines. 

Would you like me to write another Haiku for you?


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [16]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='ee52c425-4916-4afb-879e-f50212060f28'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 67, 'total_tokens': 216, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 136, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'cohere/north-mini-code:free', 'system_fingerprint': None, 'id': 'gen-1788639778-MHzrysFMVowqCz2U5WqA', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0733d-15f6-7170-9ab4-a0569b19a909-0', tool_

You can select just the last message, and you can see where the final message is coming from.

In [17]:
result["messages"][-1]

AIMessage(content="Here's a sports haiku for you:\n\nFirst strike takes aim\nWhistle blows in midnight air  \nVictory or fall\n\nThis Haiku has been checked and confirmed to have exactly 3 lines. \n\nWould you like me to write another Haiku for you?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 212, 'prompt_tokens': 78, 'total_tokens': 290, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 181, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'cohere/north-mini-code:free', 'system_fingerprint': None, 'id': 'gen-1788639782-a4ANxJY9rUAER3ZV9LVw', 'finish_reason': 'stop', 'log

In [18]:
result["messages"][-1].usage_metadata

{'input_tokens': 78,
 'output_tokens': 212,
 'total_tokens': 290,
 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'audio': 0, 'reasoning': 181}}

In [19]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 212,
  'prompt_tokens': 78,
  'total_tokens': 290,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': 0,
   'reasoning_tokens': 181,
   'rejected_prediction_tokens': None,
   'image_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': 0,
   'cached_tokens': 0,
   'video_tokens': 0},
  'cost': 0,
  'is_byok': False,
  'cost_details': {'upstream_inference_cost': 0,
   'upstream_inference_prompt_cost': 0,
   'upstream_inference_completions_cost': 0}},
 'model_provider': 'openai',
 'model_name': 'cohere/north-mini-code:free',
 'system_fingerprint': None,
 'id': 'gen-1788639782-a4ANxJY9rUAER3ZV9LVw',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [26]:
agent = create_agent(
    model="openai:minimax/minimax-m3:free",
    tools=[check_haiku_lines],
    system_prompt="You are a programmer who only writes Haiku about software engineering. You always check your work.",
)

In [27]:
result = agent.invoke({"messages": "Please write me a poem"})

for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

checking haiku, it has 3 lines:
 Silicon dreams flow,
Lines of code bloom into life—
Bugs hide in moonlight.
================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

Silicon dreams flow,
Lines of code bloom into life—
Bugs hide in moonlight.
Tool Calls:
  check_haiku_lines (call_01a073400e7f7c90a5f095b8)
 Call ID: call_01a073400e7f7c90a5f095b8
  Args:
    text: Silicon dreams flow,
Lines of code bloom into life—
Bugs hide in moonlight.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Here's a haiku for you:

*Silicon dreams flow,*
*Lines of code bloom into life—*
*Bugs hide in moonlight.*

A small poem about the life of a programmer—where creation and imperfection dance together under the 